# 09: Modellerin Karşılaştırılması

**Proje:** Framingham Kalp Çalışması verisi ile inme (stroke) tahmini

Bu notebook 8 modeli aynı veri, aynı CV katları, aynı metrikler ve aynı eşik seçim kuralıyla yan yana koyar.

**Girdi:** Her model notebook'unun kaydettiği eğitilmiş final model: `models/<kod>.joblib`. `results.csv` kullanılmaz (`.gitignore`'da olduğu için herkeste farklı olabilir). CV skorları kayıtlı modellerden **yeniden** hesaplanır, böylece tablo her zaman aynı kurallarla üretilir.

**Önemli:** Tüm model notebook'ları **aynı ortamda** (ör. hepsi Colab'da) çalıştırılmış olmalı. Farklı scikit-learn sürümünde kaydedilmiş bir model yüklenirken uyarı ya da hata verebilir.

**Bu notebook'ta yapılanlar:**
1. Kayıtlı modellerin yüklenmesi
2. CV karşılaştırması: tüm metrikler (ortalama ± std)
3. Overfitting karşılaştırması: eğitim ve CV skoru
4. Eşik seçimi: her model için %80 recall'u en az yanlış alarmla sağlayan eşik
5. Test seti karşılaştırması: yakalanan / kaçırılan vaka, yanlış alarm, ROC ve PR eğrileri
6. XAI karşılaştırması: modeller hangi özelliklere dayanıyor?
7. Avantaj / dezavantaj tablosu ve sonuç

## 1. Kurulum

In [ ]:
# shap and lime are not preinstalled on Colab
try:
    import lime
    import shap
except ImportError:
    %pip install -q shap lime

In [ ]:
import sys
from pathlib import Path

# Repo: ../src. Colab: the src/*.py files uploaded to the working directory.
SRC_DIR = next((p for p in (Path("../src"), Path("src"), Path(".")) if (p / "config.py").exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("config.py bulunamadı. Colab'da src klasöründeki .py dosyalarını sol menüden yükleyin.")
sys.path.insert(0, str(SRC_DIR.resolve()))

ROOT = SRC_DIR.resolve().parent if SRC_DIR.name == "src" else Path(".")
MODEL_DIR = ROOT / "models"
FIG_DIR = ROOT / "figures"; FIG_DIR.mkdir(exist_ok=True)

import io
import time
import warnings
from contextlib import redirect_stdout

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

from config import MAIN_METRIC, TARGET_RECALL
from preprocessing import load_data
from metrics import METRICS, compute_metrics, cv_evaluate, plot_curves, predict_score, threshold_analysis
from explain import compare_importance, shap_global

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## 2. Modellerin yüklenmesi

Model kodları ve isimleri ekibin ortak tablosundan gelir. Bir modelin dosyası yoksa (notebook'u henüz çalıştırılmamışsa) o model atlanır ve uyarı verilir. Logistic Regression notebook'u modeli `{"model": ..., "threshold": ...}` sözlüğü olarak, diğerleri doğrudan pipeline olarak kaydediyor; ikisi de desteklenir.

In [ ]:
MODELS = {
    "lr": "Logistic Regression", "nb": "Naive Bayes",
    "knn": "KNN", "svm": "SVM",
    "dt": "Decision Tree", "rf": "Random Forest",
    "xgb": "XGBoost", "lgbm": "LightGBM",
}

X_train, X_test, y_train, y_test = load_data()
print("Eğitim:", X_train.shape, f"| inme: {y_train.sum()} | Test:", X_test.shape, f"| inme: {y_test.sum()}\n")

models = {}
for code, name in MODELS.items():
    path = MODEL_DIR / f"{code}.joblib"
    if not path.exists():
        print(f"  UYARI: {path.name} yok, {name} atlandı. Önce modelin notebook'unu çalıştırın.")
        continue
    obj = joblib.load(path)
    models[name] = obj["model"] if isinstance(obj, dict) else obj
    print(f"  {name:20s} yüklendi ({path.name})")

print(f"\n{len(models)} / {len(MODELS)} model karşılaştırılacak.")

## 3. CV karşılaştırması

Her model, kendi notebook'unda seçilen ayarlarla ortak 5 katlı CV üzerinde yeniden değerlendirilir (`cv_evaluate`, sonuç dosyaya yazılmaz). Tablo `MAIN_METRIC`'e (PR-AUC) göre sıralıdır.

**Okuma notu:** Recall, precision, F1, specificity ve accuracy bu tabloda **0.5 eşiğindedir**. Modeller farklı dengeleme yöntemleri kullandığı için 0.5 her modelde farklı bir anlama gelir. Bu yüzden modeller asıl olarak eşikten bağımsız **PR-AUC** ve **ROC-AUC** ile karşılaştırılır; seçilen eşikteki recall ve yanlış alarm sayısı Bölüm 5 ve 6'dadır.

In [ ]:
cv_results = {}
start = time.time()
for name, model in models.items():
    t = time.time()
    cv_results[name] = cv_evaluate(model, X_train, y_train, model_name=name, save=False, verbose=False)
    print(f"  {name:20s} {time.time() - t:5.1f} sn")
print(f"Toplam: {time.time() - start:.0f} sn")

mean = pd.DataFrame({n: r["summary"]["cv_mean"] for n, r in cv_results.items()}).T[METRICS]
std = pd.DataFrame({n: r["summary"]["cv_std"] for n, r in cv_results.items()}).T[METRICS]
order = mean.sort_values(MAIN_METRIC, ascending=False).index
mean, std = mean.loc[order], std.loc[order]

cv_table = mean.round(3).astype(str) + " ± " + std.round(3).astype(str)
cv_table["fit_time_s"] = [round(cv_results[n]["fit_time_s"], 2) for n in order]
cv_table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
base = y_train.mean()
ax.barh(order[::-1], mean.loc[order[::-1], MAIN_METRIC], xerr=std.loc[order[::-1], MAIN_METRIC],
        color="#4C72B0", capsize=3)
ax.axvline(base, color="gray", linestyle="--", label=f"rastgele model (inme oranı = {base:.3f})")
ax.set_xlabel(f"CV {MAIN_METRIC} (hata çubuğu: katlar arası std)")
ax.set_title(f"Modellerin CV {MAIN_METRIC} karşılaştırması")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "comparison_cv_pr_auc.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(12, 4.5))
mean[["pr_auc", "roc_auc"]].plot(kind="bar", yerr=std[["pr_auc", "roc_auc"]], capsize=3, rot=20, ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("CV ortalaması")
ax.set_title("Eşikten bağımsız metrikler: PR-AUC ve ROC-AUC")
plt.tight_layout()
plt.savefig(FIG_DIR / "comparison_cv_metrics.png", dpi=200)
plt.show()

**Farklar anlamlı mı?** İki modelin PR-AUC farkı, katlar arası standart sapmadan küçükse aralarında gerçek bir fark olduğu söylenemez. Aşağıdaki tablo her modeli en iyi modelle karşılaştırır.

In [ ]:
best_name = order[0]
diff = pd.DataFrame({
    f"{MAIN_METRIC}": mean[MAIN_METRIC],
    f"en iyiden fark": mean[MAIN_METRIC] - mean.loc[best_name, MAIN_METRIC],
    "std": std[MAIN_METRIC],
})
diff["fark std'den küçük mü?"] = diff["en iyiden fark"].abs() <= diff["std"]
diff.round(4)

## 4. Overfitting karşılaştırması

Eğitim skoru CV skorundan çok yüksekse model eğitim verisini ezberliyor demektir. `cv_evaluate` eğitim skorunu SMOTE uygulanmamış gerçek eğitim katı üzerinde hesaplar.

In [ ]:
gap = pd.DataFrame({
    "eğitim": [cv_results[n]["summary"].loc[MAIN_METRIC, "train_mean"] for n in order],
    "CV": [cv_results[n]["summary"].loc[MAIN_METRIC, "cv_mean"] for n in order],
}, index=order)
gap["fark"] = gap["eğitim"] - gap["CV"]

ax = gap[["eğitim", "CV"]].plot(kind="bar", rot=20, figsize=(11, 4.5))
ax.set_ylabel(MAIN_METRIC)
ax.set_title(f"Overfitting kontrolü: eğitim ve CV {MAIN_METRIC}")
plt.tight_layout()
plt.savefig(FIG_DIR / "comparison_overfitting.png", dpi=200)
plt.show()
gap.round(3)

## 5. Eşik seçimi

Her model için eşik, notebook'larda olduğu gibi **test setine bakılmadan**, CV tahminleri (OOF) üzerinde seçilir: recall en az `TARGET_RECALL` olacak şekilde en az yanlış alarm veren eşik. Aynı fonksiyon ve aynı katlar kullanıldığı için modellerin kendi notebook'larındaki eşiklerle aynı sonuç çıkar.

**Not:** Eşik değerleri modeller arasında karşılaştırılamaz (her modelin skor ölçeği farklıdır, ör. SVM'de skor bir olasılık değil). Karşılaştırılacak olan, o eşikte kaç vakanın yakalandığı ve kaç yanlış alarm verildiğidir.

In [ ]:
thresholds, rows = {}, []
for name in order:
    r = cv_results[name]
    with redirect_stdout(io.StringIO()):          # her modelin uzun eşik tablosunu gizle
        _, thr = threshold_analysis(r["y"], r["oof_score"], model_name=name, plot=False)
    thresholds[name] = thr
    m = compute_metrics(r["y"], r["oof_score"], thr)
    pred = (r["oof_score"] >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(r["y"], pred, labels=[0, 1]).ravel()
    rows.append({"model": name, "eşik": thr, "recall": m["recall"], "precision": m["precision"],
                 "specificity": m["specificity"], "1000 kişide yanlış alarm": fp / len(pred) * 1000})

cv_threshold_table = pd.DataFrame(rows).set_index("model").round(3)
print(f"CV (OOF) tahminleri, recall >= {TARGET_RECALL:.0%} hedefiyle seçilen eşikte:")
cv_threshold_table

## 6. Test seti karşılaştırması

Kayıtlı modeller kendi notebook'larında tüm eğitim verisiyle eğitilmiştir. Burada her biri, Bölüm 5'te seçilen eşikle test setinde değerlendirilir. Test seti model seçimi için kullanılmaz; yalnızca seçilen modelin yeni veride nasıl davrandığını gösterir.

In [ ]:
test_rows, test_scores = [], {}
for name in order:
    score = predict_score(models[name], X_test)
    test_scores[name] = (y_test, score)
    thr = thresholds[name]
    m = compute_metrics(y_test, score, thr)
    tn, fp, fn, tp = confusion_matrix(y_test, (score >= thr).astype(int), labels=[0, 1]).ravel()
    test_rows.append({"model": name, "yakalanan": tp, "kaçırılan": fn, "yanlış alarm": fp,
                      "recall": m["recall"], "precision": m["precision"], "specificity": m["specificity"],
                      "f1": m["f1"], "roc_auc": m["roc_auc"], "pr_auc": m["pr_auc"],
                      "CV pr_auc": mean.loc[name, "pr_auc"]})

test_table = pd.DataFrame(test_rows).set_index("model")
test_table["CV - test pr_auc"] = test_table["CV pr_auc"] - test_table["pr_auc"]
print(f"Test seti: {len(y_test)} kişi, {int(y_test.sum())} inme. Her model kendi seçilen eşiğinde.")
test_table.round(3)

In [ ]:
plot_curves(test_scores, title="Test seti: tüm modeller")

In [ ]:
# Aynı grafiği rapor için dosyaya kaydet (plot_curves dosyaya yazmadığı için)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score
for name, (y, s) in test_scores.items():
    fpr, tpr, _ = roc_curve(y, s); p, r, _ = precision_recall_curve(y, s)
    axes[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y, s):.3f})")
    axes[1].plot(r, p, label=f"{name} ({average_precision_score(y, s):.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[1].axhline(np.mean(y_test), color="gray", linestyle="--", label="rastgele")
axes[0].set(xlabel="Yanlış alarm oranı (1 - specificity)", ylabel="Recall", title="ROC eğrileri (test)")
axes[1].set(xlabel="Recall", ylabel="Precision", title="PR eğrileri (test)")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "comparison_test_curves.png", dpi=200)
plt.close()
print("Kaydedildi: figures/comparison_test_curves.png")

In [ ]:
# Yakalanan / kaçırılan / yanlış alarm: sunum için tek bakışta
ax = test_table[["yakalanan", "kaçırılan", "yanlış alarm"]].plot(kind="bar", rot=20, figsize=(11, 4.5),
                                                                  color=["#55A868", "#C44E52", "#DD8452"])
ax.set_ylabel("kişi sayısı (test seti)")
ax.set_title(f"Test seti, seçilen eşikte: {int(y_test.sum())} inme vakası, {len(y_test)} kişi")
for c in ax.containers:
    ax.bar_label(c, fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "comparison_test_counts.png", dpi=200)
plt.show()

## 7. XAI karşılaştırması: modeller hangi özelliklere dayanıyor?

Her modelin SHAP önemleri hesaplanır ve `compare_importance` ile yüzdeye çevrilir (modellerin SHAP ölçekleri farklı olduğu için ham değerler karşılaştırılmaz). Tüm modellerin ortak olarak öne çıkardığı özellikler, inme riski için en güvenilir göstergeler olarak yorumlanabilir.

KNN, SVM ve Naive Bayes için yavaş KernelExplainer kullanıldığından örneklem sınırlıdır (100 hasta, 20 küme merkezi, hasta başına 300 kombinasyon). Toplam birkaç dakika sürer.

In [ ]:
xai = {}
start = time.time()
for name in order:
    t = time.time()
    imp = shap_global(models[name], X_train, X_test, model_name=name, plot=False,
                      sample_size=100, background_size=20, kernel_nsamples=300)
    xai[name] = {"shap_importance": imp}
    print(f"  {name:20s} {time.time() - t:5.1f} sn")
print(f"Toplam: {time.time() - start:.0f} sn")

shares = compare_importance(xai)
shares["ortalama"] = shares.mean(axis=1)
shares = shares.sort_values("ortalama", ascending=False)
shares.head(12).round(3)

In [ ]:
top = shares.head(12).drop(columns="ortalama")
fig, ax = plt.subplots(figsize=(1.1 * len(top.columns) + 4, 6))
im = ax.imshow(top.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(top.columns)), top.columns, rotation=30, ha="right")
ax.set_yticks(range(len(top.index)), top.index)
for i in range(top.shape[0]):
    for j in range(top.shape[1]):
        v = top.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                color="white" if v > top.values.max() * 0.6 else "black")
fig.colorbar(im, ax=ax, label="SHAP önem payı")
ax.set_title("Modellere göre en önemli 12 özellik (SHAP önem payı)")
plt.tight_layout()
plt.savefig(FIG_DIR / "comparison_shap_heatmap.png", dpi=200)
plt.show()

In [ ]:
# Her modelin ilk 5 özelliği: sunumda "modeller aynı şeylere mi bakıyor?" sorusu için
pd.DataFrame({name: xai[name]["shap_importance"]["feature"].head(5).tolist() for name in order},
             index=[f"{i}." for i in range(1, 6)])

## 8. Modellerin avantajları ve dezavantajları

| Model | Aile | Avantajları | Dezavantajları |
|---|---|---|---|
| **Logistic Regression** | Lineer | Basit, hızlı, katsayıları doğrudan yorumlanabilir; tıpta risk modellerinin standardı | Yalnızca doğrusal ilişkileri yakalar; etkileşimleri elle eklemek gerekir |
| **Naive Bayes** | Olasılıksal | Çok hızlı, az veriyle çalışır | Özelliklerin birbirinden bağımsız olduğunu varsayar; burada tansiyon türevleri birbirine çok bağlı |
| **KNN** | Mesafe | Basit, varsayım yapmaz | Tahmin yavaş; ölçeğe ve gereksiz sütunlara duyarlı; dengesiz veride zayıf |
| **SVM** | Sınır | Marjin sayesinde overfitting'e dirençli; kernel ile doğrusal olmayan sınırlar | Büyük veride yavaş; skoru olasılık değil; kara kutu |
| **Decision Tree** | Ağaç | Kuralları çizilip okunabilir | Tek başına kararsız ve overfitting'e yatkın |
| **Random Forest** | Bagging | Kararlı, ölçekleme gerektirmez, etkileşimleri yakalar | Tek ağaç gibi okunamaz; daha yavaş |
| **XGBoost** | Boosting | Tablo verisinde genelde en güçlü yöntemlerden; düzenlileştirme seçenekleri | Çok hiperparametre; küçük veride kolay overfitting |
| **LightGBM** | Boosting | XGBoost'a benzer başarı, daha hızlı | Küçük veride yaprak sayısı dikkatle sınırlanmalı |

## 9. Sonuç (rapor için)

Aşağıdaki maddeleri yukarıdaki tablolara bakarak doldurun:

- **En iyi model (CV PR-AUC):** ... (PR-AUC = ... ± ...). Rastgele bir modelin PR-AUC'si inme oranına, yaklaşık ..., eşittir.
- **Farklar anlamlı mı?** Bölüm 3'teki tabloya göre ... modelin en iyiden farkı katlar arası standart sapmadan küçük; bu modeller arasında gerçek bir fark olduğu söylenemez.
- **Overfitting:** En büyük eğitim-CV farkı ... modelinde (...). ...
- **Test seti:** Seçilen eşikte en iyi model ... vakanın ...'ini yakalıyor, ... yanlış alarm veriyor. CV ile test arasındaki fark ... (genelleme ...).
- **Neden bazı modeller daha iyi?** ...
- **Ortak risk faktörleri (SHAP):** Tüm modellerde en çok öne çıkan özellikler: .... Bu, inme için bilinen tıbbi risk faktörleriyle (yaş, tansiyon, ...) uyumludur / değildir.
- **Kısıtlar:** Veri seti küçük ve dengesiz; inme vakası az. %80 recall hedefi, test setinde yaklaşık her ... kişiden birine yanlış alarm anlamına geliyor.